# RQ VI: Goodreads popularity


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reported_runs/analysis").is_dir())
sys.path.insert(0, str(repo)) if str(repo) not in sys.path else None
from reported_runs.analysis.presentation import load_results, NotebookReport, temporal_table, effect_table, rank_summary, unique_trajectories, label_conditions
from reported_runs.analysis.plotting import trajectory_grid, effect_grid, rank_grid, association_grid
from reported_runs.analysis.comparisons import summarise_iterations
%matplotlib inline

results = load_results()
report = NotebookReport(results, 'rq6')
scales = report.scales
trajectory = label_conditions(results["trajectory"])
sources = results["sources"]
summaries = results["summaries"]
effect_summaries = results["effect_summaries"]
pd.set_option("display.max_columns", 18)
pd.set_option("display.max_rows", 60)

# RQ VI — Popularity findings in Goodreads


In [ ]:
from reported_runs.analysis.sections_rq6_7 import (POPULARITY, HEAD_GAPS, transfer_effects, transfer_plot_labels, final_goodreads_conditions, reference_table, computational_feedback_decomposition)
rq_sources = sources.loc[sources.study.isin(['movielens_feedback_a300_e804', 'goodreads_gender_a300_e804'])]
report.availability(rq_sources)

## Frozen popularity references


In [ ]:
references = results['references'].loc[results['references'].study.isin(rq_sources.study.unique())]
context = reference_table(references)
report.table(context, 'rq6_popularity_references', "", columns=['dataset', 'seed', 'population_seed', 'catalogue_size', 'head_count', 'head_catalogue_share', 'initial_training_head_share'])

## Head cut-off ties


In [ ]:
from reported_runs.analysis.head_ties import load_goodreads_head_ties
from reported_runs.analysis.head_tie_views import head_tie_table, TABLE_NAME, TABLE_COLUMN_FORMAT, SUMMARY_EXPORT_NAME, ITEMS_EXPORT_NAME

head_summary, strict_head_items = load_goodreads_head_ties(rq_sources, repo)
head_numeric_csv = report.exports / 'data' / SUMMARY_EXPORT_NAME
head_items_csv = report.exports / 'data' / ITEMS_EXPORT_NAME
head_numeric_csv.parent.mkdir(parents=True, exist_ok=True)
head_summary.to_csv(head_numeric_csv, index=False)
strict_head_items.to_csv(head_items_csv, index=False)
report.table(head_tie_table(head_summary), TABLE_NAME, "", source_frame=rq_sources[rq_sources.dataset.eq('goodreads')], column_format=TABLE_COLUMN_FORMAT)
display(Markdown(f'[numeric CSV]({head_numeric_csv.as_posix()}) · [item CSV]({head_items_csv.as_posix()})'))

## Goodreads trajectories


In [ ]:
goodreads = final_goodreads_conditions(trajectory)
figure = trajectory_grid(goodreads, POPULARITY, title='Goodreads popularity trajectories', scales=scales)
report.figure(figure, goodreads, 'rq6_goodreads_popularity_trajectories', "")

In [ ]:
gr_summaries = summaries.loc[summaries.study.eq('goodreads_gender_a300_e804')]
table = temporal_table(gr_summaries, POPULARITY)
report.table(table, 'rq6_goodreads_popularity_summaries', "")

## Matched changes across datasets


In [ ]:
transfer = transfer_effects(effect_summaries)
mean_transfer = transfer.loc[transfer.window.eq('mean_1_30')]
figure = effect_grid(mean_transfer, POPULARITY, title='Mean matched popularity changes across datasets', scales=scales, facet='dataset', value='value')
report.figure(figure, mean_transfer, 'rq6_cross_dataset_popularity_effects', "", recommended=False, why="")

In [ ]:
table = effect_table(transfer, 'head_click_share')
report.table(table, 'rq6_cross_dataset_head_click_effects', "")

In [ ]:
exposure_transfer = transfer_plot_labels(transfer)
figure = effect_grid(exposure_transfer, ['head_exposure', 'weighted_head_exposure'], title='Cross-dataset exposure effects: mean and endpoint', scales=scales, category='dataset_comparison', facet='window', value='value')
report.figure(figure, exposure_transfer, 'rq6_cross_dataset_exposure_mean_endpoint', "")

## Selection relative to exposure


In [ ]:
figure = trajectory_grid(goodreads, HEAD_GAPS, title='Goodreads click-exposure gaps', scales=scales)
report.figure(figure, goodreads, 'rq6_goodreads_click_exposure_gaps', "")

In [ ]:
figure = effect_grid(mean_transfer, HEAD_GAPS, title='Mean matched gap changes across datasets', scales=scales, facet='dataset', value='value')
report.figure(figure, mean_transfer, 'rq6_cross_dataset_gap_effects', "")

## Clicking, coverage, and concentration


In [ ]:
figure = trajectory_grid(goodreads, ['click_rate', 'no_click_rate', 'valid_clicks'], title='Goodreads clicking behaviour', scales=scales)
report.figure(figure, goodreads, 'rq6_goodreads_click_behaviour', "")

In [ ]:
figure = trajectory_grid(goodreads, ['exposure_coverage', 'click_coverage', 'exposure_gini', 'click_gini'], title='Goodreads coverage and concentration', scales=scales)
report.figure(figure, goodreads, 'rq6_goodreads_coverage_concentration', "")

In [ ]:
table = temporal_table(gr_summaries, ['click_rate', 'no_click_rate', 'valid_clicks'])
report.table(table, 'rq6_goodreads_click_behaviour_summaries', "")

In [ ]:
table = temporal_table(gr_summaries, ['exposure_coverage', 'click_coverage', 'exposure_gini', 'click_gini'])
report.table(table, 'rq6_goodreads_coverage_concentration_summaries', "")

## Supplementary L1 decomposition


In [ ]:
decomposition = computational_feedback_decomposition(trajectory, sources)
figure = effect_grid(decomposition, POPULARITY, title='Goodreads computational effects: L1 minus L0 and L2 minus L1', scales=scales, facet='window', value='value')
report.figure(figure, decomposition, 'rq6_computational_l1_decomposition', "")

In [ ]:
table = effect_table(decomposition, 'head_click_share')
report.table(table, 'rq6_computational_l1_decomposition_table', "")

## Descriptive mean and consistency across available seeds


In [ ]:
from reported_runs.analysis.presentation import cross_seed_table
for metric in ['head_click_share', 'weighted_head_exposure']:
    table = cross_seed_table(results, 'rq6', [metric])
    report.table(table, f'rq6_seed_consistency_{metric}', "")